# History feature modls comparison

Core logic same as 3 fatigue_modeling.ipynb.

In [1]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Ensure local src edits are picked up when re-running this cell.
for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

import pandas as pd
from modeling.baselines import (
    run_all_baseline_benchmarks,
    summarize_baseline_metrics,
)
from modeling.config import (
    DATA_PATH,
    HISTORY_CANDIDATE_FEATURES,
    HISTORY_FEATURES,
    N_CV_FOLDS,
    OPTUNA_TRIALS,
    TIME_COL,
    TIME_SERIES_GROUP_COLS,
)
from modeling.data import (
    build_split_bundle,
    load_fatigue_data,
    participant_strata,
    preprocess_after_split,
    split_participant_ids,
    split_summary_table,
)
from modeling.registry import ORDINAL_MODELS
from modeling.runner import tune_and_benchmark_model
from modeling.significance import compare_history_feature_count_significance
from modeling.summaries import (
    CATEGORY_ORDER,
    build_history_feature_count_comparison,
    collect_categorized_summaries,
    collect_summaries,
)

HISTORY_7_COLS = list(HISTORY_CANDIDATE_FEATURES)
HISTORY_3_COLS = list(HISTORY_FEATURES)

In [3]:
# Load and preprocess data, then split into train/val/test sets.
df = load_fatigue_data('../../' + DATA_PATH)
df = df.sort_values(TIME_SERIES_GROUP_COLS + [TIME_COL]).reset_index(drop=True)

strata = participant_strata(df)
train_val_ids, test_ids = split_participant_ids(df['id'].unique(), strata=strata)
train_val_mask = df['id'].isin(train_val_ids)
test_mask = df['id'].isin(test_ids)

literacy_col = 'menstrual_health_literacy_num'
print(f'Literacy NaNs before preprocess: {df[literacy_col].isna().sum()}')
df = preprocess_after_split(df, train_val_mask)
print(f'Literacy NaNs after preprocess: {df[literacy_col].isna().sum()}')

bundle = build_split_bundle(df, train_val_ids, test_ids, train_val_mask, test_mask)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle))
print('Test participant ids:', sorted(bundle.test_ids))


Literacy NaNs before preprocess: 80
Literacy NaNs after preprocess: 0
Rows: 3,331  Participants: 42


,split,participants,rows,mean_fatigue
0,train_val,34,2659,2.462204
1,test,8,672,2.653274


Test participant ids: [np.int64(7), np.int64(14), np.int64(24), np.int64(38), np.int64(40), np.int64(41), np.int64(46), np.int64(50)]


Re-run the **init accumulators** cell below before a fresh partial run to clear prior tuned-model results.


In [4]:
# Re-run this cell to clear accumulated model results before a fresh partial run.
ordinal_results = []
history7_ordinal_results = []
history3_ordinal_results = []

ordinal_best_params = {}
history7_best_params = {}
history3_best_params = {}

In [5]:
# Baseline benchmarks (no history features) are run first, so that tuned models can be compared to them.
ordinal_baseline_results = run_all_baseline_benchmarks(bundle, n_splits=N_CV_FOLDS)

ordinal_baseline_summary = summarize_baseline_metrics(ordinal_baseline_results)

print('Ordinal baselines (test metrics)')
display(ordinal_baseline_summary[[c for c in ordinal_baseline_summary.columns if c.startswith('test_')]])


Ordinal baselines (test metrics)


,test_mae,test_rmse,test_r2,test_qwk
model,,,,
global_mean,1.406250,1.640721,-0.188402,0.000000
global_mode,1.156250,1.544479,-0.053072,0.000000
lag1_fatigue,0.950893,1.424175,0.104593,0.549449
expanding_mean,1.025298,1.336863,0.211017,0.422289


MAE: Mean Absolute Error;

RMSE: Root Mean Squared Error, measures the variation in residual/error

R2: how much variability is explained by the model

QWK: Quadratic Weighted Kappa. QWK measures the agreement between two raters—such as an AI and a human—on an ordered scale. It is designed to adjust for chance agreements and heavily penalize larger scoring discrepancies over minor ones.

### History (7 features)

Same seven ordinal models as above, with **all seven history candidate columns** (`HISTORY_CANDIDATE_FEATURES`) appended to the daily feature matrix. History construction uses `EWMA_ALPHA` and `ROLLING_WINDOWS` from `config.py` via `build_split_bundle`; first-day NaNs in history columns are imputed with the train/val median.
> The `EWMA_ALPHA` and `ROLLING_WINDOWS` are optimized from the [`history feature engineering.ipynb`](history%20feature%20engineering.ipynb) notebook.

**History features** (7 cols):
- fatigue lag1: Yesterday's fatigue score
- fatigue EWMA: Exponentially weighted average of past fatigue; recent days count more
- fatigue expanding mean: Average fatigue on all earlier days for this person
- fatigue delta lag1: Change in fatigue, the worsening/improving trend
- activity_logsum_roll_mean: Rolling mean of prior days' sum of log1p(lightly) + log1p(moderately) + log1p(very) (window from ROLLING_WINDOWS)
- calories_sum_roll_mean: Rolling mean of prior daily calories burned (window from ROLLING_WINDOWS)
- very_active_roll_mean: Rolling mean of prior "very active" minutes (window from ROLLING_WINDOWS)


#### Ordinal Regression (history7)

Continuous loss on `fatigue_num`, then round and clip to [0, 5].


##### `linear_regression` (history7)


In [6]:
_name = 'linear_regression'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_7_COLS,
    display_name=f'{_name}_history7',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history7_ordinal_results.append(_result)
history7_best_params[f'{_name}_history7'] = _params
print(f'[ok] {_name}_history7  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] linear_regression_history7  test_mae=0.9296


##### `ordinal_rf` (history7)


In [7]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_7_COLS,
    display_name=f'{_name}_history7',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history7_ordinal_results.append(_result)
history7_best_params[f'{_name}_history7'] = _params
print(f'[ok] {_name}_history7  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf_history7  test_mae=0.9450


##### `catboost_regressor` (history7)


In [8]:
_name = 'catboost_regressor'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_7_COLS,
    display_name=f'{_name}_history7',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history7_ordinal_results.append(_result)
history7_best_params[f'{_name}_history7'] = _params
print(f'[ok] {_name}_history7  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_regressor_history7  test_mae=0.9480


### History (3 features)

Same seven ordinal models with the **FE forward-selected 3-column subset** (`HISTORY_FEATURES` from `config.py`: `fatigue_ewma`, `fatigue_expanding_mean`, `fatigue_lag1`). Same construction params as the 7-feature block above.


#### Ordinal Regression (history3)

Continuous loss on `fatigue_num`, then round and clip to [0, 5].


##### `linear_regression` (history3)


In [9]:
_name = 'linear_regression'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_3_COLS,
    display_name=f'{_name}_history3',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history3_ordinal_results.append(_result)
history3_best_params[f'{_name}_history3'] = _params
print(f'[ok] {_name}_history3  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] linear_regression_history3  test_mae=0.9312


##### `ordinal_rf` (history3)


In [10]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_3_COLS,
    display_name=f'{_name}_history3',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history3_ordinal_results.append(_result)
history3_best_params[f'{_name}_history3'] = _params
print(f'[ok] {_name}_history3  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf_history3  test_mae=0.9448


##### `catboost_regressor` (history3)


In [11]:
_name = 'catboost_regressor'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_3_COLS,
    display_name=f'{_name}_history3',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history3_ordinal_results.append(_result)
history3_best_params[f'{_name}_history3'] = _params
print(f'[ok] {_name}_history3  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_regressor_history3  test_mae=0.9377


### Results summary

Aggregates baselines plus any models ran (`_history7`, and/or `_history3`). Results are grouped into **baseline** and **history** categories.

The next cell prints **CV** tables in order baseline → history, then **test** tables in the same order. Within each table, rows are sorted by `cv_mae` or `test_mae` respectively. The following cells compare 3-col vs 7-col history test MAE.

In [12]:
# Merge baselines (§2), base tuned models (§3), and history variants (§3 History).
# globals().get(...) allows partial notebook runs without NameError on skipped cells.

history7_ordinal_results = globals().get('history7_ordinal_results', [])
history3_ordinal_results = globals().get('history3_ordinal_results', [])
history7_best_params = globals().get('history7_best_params', {})
history3_best_params = globals().get('history3_best_params', {})

ran_tuned_models = sorted(
    set(history7_best_params)
    | set(history3_best_params)
)
print(f'Ran {len(ran_tuned_models)} tuned ordinal models: {ran_tuned_models}')

all_ordinal_results = (
    ordinal_baseline_results
    + history7_ordinal_results
    + history3_ordinal_results
)

ordinal_cv_summary, ordinal_test_summary = collect_summaries(all_ordinal_results)
category_summaries = collect_categorized_summaries(all_ordinal_results)

print('CV (sorted by cv_mae within each category; cv_* = mean over GroupKFold folds on train/val)')
for category in CATEGORY_ORDER:
    cv_cat, _ = category_summaries[category]
    if cv_cat.empty:
        continue
    print(f'  {category}')
    display(cv_cat)

print('Test (sorted by test_mae within each category; refit on full train/val, scored on test participants)')
for category in CATEGORY_ORDER:
    _, test_cat = category_summaries[category]
    if test_cat.empty:
        continue
    print(f'  {category}')
    display(test_cat)

Ran 6 tuned ordinal models: ['catboost_regressor_history3', 'catboost_regressor_history7', 'linear_regression_history3', 'linear_regression_history7', 'ordinal_rf_history3', 'ordinal_rf_history7']
CV (sorted by cv_mae within each category; cv_* = mean over GroupKFold folds on train/val)
  baseline


,best_params,cv_mae,cv_rmse,cv_r2,cv_qwk,cv_mae_std
model,,,,,,
lag1_fatigue,{},0.824015,1.315205,0.096947,0.546287,0.193110
expanding_mean,{},0.867974,1.198179,0.280799,0.495990,0.127344
global_mode,{},1.216046,1.554387,-0.200378,0.000000,0.243084
global_mean,{},1.348516,1.606173,-0.297646,0.000000,0.153930


  history


,best_params,cv_mae,cv_rmse,cv_r2,cv_qwk,cv_mae_std
model,,,,,,
ordinal_rf_history3,"{'n_estimators': 380, 'max_depth': 3, 'min_sam...",0.849683,1.098049,0.384677,NaN,0.124348
ordinal_rf_history7,"{'n_estimators': 380, 'max_depth': 3, 'min_sam...",0.850374,1.098401,0.384294,NaN,0.124920
catboost_regressor_history3,"{'iterations': 345, 'depth': 4, 'learning_rate...",0.870929,1.106011,0.374613,NaN,0.113906
catboost_regressor_history7,"{'iterations': 345, 'depth': 4, 'learning_rate...",0.871807,1.106990,0.374343,NaN,0.109052
linear_regression_history7,{'alpha': 9.668599030214608},0.875630,1.118312,0.353944,NaN,0.120267
linear_regression_history3,{'alpha': 0.9675732510483003},0.878006,1.120835,0.351043,NaN,0.117872


Test (sorted by test_mae within each category; refit on full train/val, scored on test participants)
  baseline


,best_params,test_mae,test_rmse,test_r2,test_qwk
model,,,,,
lag1_fatigue,{},0.950893,1.424175,0.104593,0.549449
expanding_mean,{},1.025298,1.336863,0.211017,0.422289
global_mode,{},1.156250,1.544479,-0.053072,0.000000
global_mean,{},1.406250,1.640721,-0.188402,0.000000


  history


,best_params,test_mae,test_rmse,test_r2,test_qwk
model,,,,,
linear_regression_history7,{'alpha': 9.668599030214608},0.929633,1.199076,0.365273,NaN
linear_regression_history3,{'alpha': 0.9675732510483003},0.931197,1.201181,0.363042,NaN
catboost_regressor_history3,"{'iterations': 345, 'depth': 4, 'learning_rate...",0.937652,1.198162,0.366240,NaN
ordinal_rf_history3,"{'n_estimators': 380, 'max_depth': 3, 'min_sam...",0.944795,1.212132,0.351374,NaN
ordinal_rf_history7,"{'n_estimators': 380, 'max_depth': 3, 'min_sam...",0.945021,1.212257,0.351241,NaN
catboost_regressor_history7,"{'iterations': 345, 'depth': 4, 'learning_rate...",0.947952,1.207689,0.356121,NaN


In [13]:
# --- 3-col vs 7-col history comparison (test MAE only) ---
# delta_mae_3_minus_7 = history3 - history7; negative means 3-col subset wins.

history_count_comparison = build_history_feature_count_comparison(
    ordinal_test_summary, ORDINAL_MODELS
)
if history_count_comparison.empty:
    print('No paired history7/history3 models found — run both history blocks first.')
else:
    print(
        '3-col vs 7-col history comparison '
        '(delta_mae_3_minus_7 = history3 - history7; negative = 3-col better)'
    )
    display(history_count_comparison)

3-col vs 7-col history comparison (delta_mae_3_minus_7 = history3 - history7; negative = 3-col better)


,test_mae_history7,test_mae_history3,delta_mae_3_minus_7
model,,,
catboost_regressor,0.947952,0.937652,-0.010300
ordinal_rf,0.945021,0.944795,-0.000226
linear_regression,0.929633,0.931197,0.001564


### Significance of 3-col vs 7-col difference

The table above reports aggregate held-out test MAE. This section tests whether the **per-participant** MAE difference is larger than sampling noise.

**Method:** For each model, refit `history7` and `history3` with their separate Optuna params, predict on the same test rows, compute per-participant `delta = MAE3 - MAE7`, then run a two-sided Wilcoxon signed-rank test and a participant-level bootstrap 95% CI on the mean delta. Benjamini-Hochberg FDR adjusts p-values across all 7 models.

**Where the uncertainty comes from:** The p-values and CIs are not from re-running Optuna or re-splitting data. They reflect noise in the **fixed held-out test set** (~8 participants): day-to-day fatigue variability, imperfect predictions, and the fact that these eight people are only one sample from 42. Even if 3-col and 7-col were equally good, the eight per-participant deltas would rarely all be exactly zero — they can accidentally lean toward one side. With only eight clusters, a mildly one-sided pattern can happen by luck; that is what a high p-value means.

**Why `fdr_p` is used (multiple comparisons):** We run one significance test per model family (7 tests total). If we called any `wilcoxon_p < 0.05` "significant" without correction, we would expect roughly one false alarm among seven even when 3-col and 7-col are equal everywhere (~30% chance of at least one lucky small p). `fdr_p` adjusts each raw p-value for those seven tries so a low adjusted value is less likely to be a fluke from testing many models. Prefer **`fdr_p`** over **`wilcoxon_p`** when deciding which models show a reliable difference.

**How to read the columns:**

| Column | Interpretation |
|---|---|
| `delta_mae_3_minus_7` | Overall held-out test MAE for 3-col minus 7-col (all test days pooled). **Negative = 3-col wins** on average test error. Matches the table above. |
| `mean_participant_delta` | Average of per-participant `(MAE3 - MAE7)`. Same sign convention; less dominated by participants with many test days. |
| `ci_low`, `ci_high` | Bootstrap 95% CI for `mean_participant_delta` (participants resampled with replacement). If the interval excludes 0, the average per-participant difference is more clearly directional; if 0 is inside, the difference is uncertain. |
| `wilcoxon_p` | Two-sided Wilcoxon signed-rank p-value on per-participant deltas. Low values (below 0.05) suggest the paired differences are unlikely under “no difference”, thus suggests true difference/statistically significant. Does not by itself quantify effect size. |
| `fdr_p` | Benjamini-Hochberg adjusted p-value across all 7 models. Use this (not raw `wilcoxon_p`) when judging significance after multiple comparisons. Low value (eg. 0.01, 0.03) indicates unlikely by pure luck, thus "statistically significant". High value (eg. 0.4, 0.8) indicates likely to be noise. |

**Caveats:**
- Separate Optuna tuning means a significant result reflects **held-out performance difference**, not a pure feature-count effect (hyperparameter differences are included).
- With ~8 held-out test participants, power is limited; small MAE gaps may be real but not reach `p < 0.05`.
- Negative `mean_participant_delta` or `delta_mae_3_minus_7` means the 3-col subset wins.

In [14]:
# --- Participant-level significance: 3-col vs 7-col history ---
# Requires history7_best_params and history3_best_params from both history blocks.

significance_summary = compare_history_feature_count_significance(
    bundle,
    ORDINAL_MODELS,
    history7_best_params,
    history3_best_params,
    history7_cols=HISTORY_7_COLS,
    history3_cols=HISTORY_3_COLS,
)
if significance_summary.empty:
    print('No paired history7/history3 params found — run both history blocks first.')
else:
    print(
        'Participant-level 3-col vs 7-col significance '
        '(negative mean_participant_delta = 3-col better; wilcoxon_p two-sided; fdr_p = BH across models)'
    )
    display(significance_summary)

Participant-level 3-col vs 7-col significance (negative mean_participant_delta = 3-col better; wilcoxon_p two-sided; fdr_p = BH across models)


,delta_mae_3_minus_7,mean_participant_delta,ci_low,ci_high,wilcoxon_p,fdr_p
model,,,,,,
ordinal_rf,0.000000,0.000000,0.000000,0.000000,NaN,1.0
catboost_regressor,0.000000,-0.000694,-0.019194,0.014241,0.84375,1.0
linear_regression,0.013393,0.012127,-0.011753,0.034651,0.37500,1.0
